#### Define notebook Parameters

Pass through th pl_worker

In [58]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
task_executions_id = ""
parent_run_id = ""
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
#prev_wm = "1900-01-01 00:00:00"
prev_wm = "2026-06-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"
server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"

# Framework-style placeholder; real connection settings should come from pipeline/metadata
source_connection_settings = "{}"

# Source config: one database, many tables to load
source_settings = json.dumps({
    "source_name": "CAN_RMDD",
    "database_name": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
    "table_config": [
        {
            "schema_name": "dbo",
            "table_name": "Address",
            "target_table": "silver_rmdd_address",
            "primary_keys": ["AddressID"],
            # "select_columns": [
            #     "AddressID",
            #     "AddressTypeCode",
            #     "Address1",
            #     "Address2",
            #     "Address3",
            #     "Address4",
            #     "Address5",
            #     "City",
            #     "CountryID",
            #     "State",
            #     "PostalCode",
            #     "Active",
            #     "LastUpdatedDateGMT"
            # ],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "Client",
            "target_table": "silver_rmdd_client",
            "primary_keys": ["ClientNumber", "MemberFirmCode"],
            # "select_columns": [
            #     "ClientID",
            #     "ClientNumber",
            #     "MemberFirmCode",
            #     "PMSClientID",
            #     "ClientName",
            #     "AlternateClientName",
            #     "ContactName",
            #     "OpenDateGMT",
            #     "CloseDateGMT",
            #     "ClientDUNS",
            #     "ClientGUDUNS",
            #     "Comments",
            #     "PMSEntityType",
            #     "ClientType",
            #     "IsConfidential",
            #     "SICCode",
            #     "SanctionsChecked_YN",
            #     "Active",
            #     "CreatedDateGMT",
            #     "LastUpdatedDateGMT"
            # ],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "Person",
            "target_table": "silver_rmdd_person",
            "primary_keys": ["SystemEmployeeCode", "MemberFirmCode"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        }
    ]
})

# Target config: common silver destination settings
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_rmdd",
    "load_strategy": "merge",
    "location_root": "Files/silver/rmdd"
})

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 84, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [59]:
%run nb_transactional_shared_functions

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 89, Finished, Available, Finished, True)

In [60]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 90, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [61]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("database_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge")
location_root = target_settings.get("location_root")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
)

if not server_name: server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
print(source_name)
print(source_database)
print(target_schema)
display(table_config)

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 91, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd


SynapseWidget(Synapse.DataFrame, a1c3b6fc-5763-4bb6-b5a2-087dc0a35976)

#### Build JDBC connection

Connects to source SQL database

In [62]:
# Build one JDBC connection for the source SQL database
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(jdbc_url)
print(connection_properties)

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 92, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;
{'accessToken': '[REDACTED]', 'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'}


#### Ingest source table

Read and dedup clean source

In [63]:
# Read each source table and drop exact duplicates only
for cfg in table_config:
    # Get variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    column_list = cfg.get("select_columns", ["*"])
    is_incremental = cfg.get("is_incremental", 0)
    watermark_col = cfg.get("incremental_column")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    column_list_sql = ", ".join(column_list)
    source_view = f"src_{source_table.lower()}"

    # Set watermark based on YAML watermark_col
    watermark_filter = "1 = 1"
    if is_incremental == 1:
        watermark_filter = f"{watermark_col} > '{prev_wm}' AND {watermark_col} <= '{curr_wm}'"

    # Query to source with watermark applied
    source_query = f"""
    (
        SELECT {column_list_sql}
        FROM {full_source_name}
        WHERE {watermark_filter}
    ) AS src
    """

    # Read table using JDBC connection
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    # Remove duplicates and create temp view for later consumption
    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)

    print(f"Source preview for {full_source_name}")
    display(df_source.limit(3))

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 93, Finished, Available, Finished, False)

Source preview for dbo.Address


SynapseWidget(Synapse.DataFrame, 3e199783-954a-4691-8535-d1abab147234)

Source preview for dbo.Client


SynapseWidget(Synapse.DataFrame, 5202defb-9b31-44ea-99fd-9b3e8c378a96)

Source preview for dbo.Person


SynapseWidget(Synapse.DataFrame, 9e71ccff-1350-49cb-adac-0e51c50e939c)

#### Transform source table

Applies mapping and transformation as needed

In [64]:
# Map source columns into the target shape
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    source_view = f"src_{source_table.lower()}"
    target_view = f"tgt_{source_table.lower()}"

    # Transformation block
    if source_table == "Address":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY AddressID) AS BIGINT) AS address_key,
            AddressID               AS address_id,
            AddressTypeCode         AS address_type_code,
            Address1                AS address_1,
            Address2                AS address_2,
            Address3                AS address_3,
            Address4                AS address_4,
            Address5                AS address_5,
            City                    AS city,
            CountryID               AS country_id,
            CAST(NULL AS STRING)    AS country_code_iso2,
            State                   AS state,
            PostalCode              AS zip_code,
            Address1                AS street_address_check,
            Active                  AS is_active,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "Client":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY ClientID) AS BIGINT) AS client_key,
            ClientNumber            AS client_number,
            MemberFirmCode          AS member_firm_code,
            ClientID                AS client_id,
            PMSClientID             AS pms_client_id,
            ClientName              AS client_name,
            AlternateClientName     AS alternate_client_name,
            ContactName             AS contact_name,
            OpenDateGMT             AS open_date_utc,
            CloseDateGMT            AS close_date_utc,
            ClientDUNS              AS client_duns,
            ClientGUDUNS            AS client_guduns,
            Comments                AS comments,
            PMSEntityType           AS pms_entity_type,
            ClientType              AS client_type,
            IsConfidential          AS is_confidential,
            SICCode                 AS sic_code,
            CAST(NULL AS STRING)    AS client_sector_key,
            SanctionsChecked_YN     AS is_sanctions_checked,
            Active                  AS is_active,
            CreatedDateGMT          AS created_date_utc_at_source,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "Person":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY PersonID) AS BIGINT) AS person_key,
            SystemEmployeeCode              AS person_number,
            MemberFirmCode                  AS member_firm_code,
            PersonID                        AS person_id,
            PersonStatusCode                AS person_status_code,
            Gender                          AS gender,
            Prefix                          AS prefix,
            FirstName                       AS first_name,
            FirstNameKnownAs                AS first_name_known_as,
            MiddleName                      AS middle_name,
            LastName                        AS last_name,
            Suffix                          AS suffix,
            Initials                        AS initials,
            PhotoPath                       AS photo_path,
            CommencementDateGMT             AS commencement_date_utc,
            DepartureDateGMT                AS departure_date_utc,
            LeaveStatus                     AS leave_status,
            SubTeamID                       AS team_key,
            JobTitle                        AS job_title,
            JobTitlePMS                     AS job_title_pms,
            PracticeGroup                   AS practice_group_code,
            CAST(NULL AS STRING)            AS practice_group_key,
            CAST(YearOfCall AS STRING)      AS year_of_call,
            LastName                        AS employee_name_reporting,
            CAST(DeleteFlag AS  BOOLEAN)    AS is_active,
            LastUpdatedDateGMT              AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    else:
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT *
        FROM {source_view}
        """)

    print(f"Mapped preview for {target_view}")
    display(spark.table(target_view).limit(3))

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 94, Finished, Available, Finished, False)

Mapped preview for tgt_address


SynapseWidget(Synapse.DataFrame, 2084b91d-626d-43f1-ae47-e99b06960ef0)

Mapped preview for tgt_client


SynapseWidget(Synapse.DataFrame, 7fb4a2fc-ec8d-4484-ac74-9e24014a1075)

Mapped preview for tgt_person


SynapseWidget(Synapse.DataFrame, f2e177d1-4efc-44ea-8430-552c372bfffd)

#### Add metadata to source table

Applies metadata and _hashed_pk

In [65]:
# Add metadata and hashed key to the transformed tgt_* views
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    target_view = f"tgt_{source_table.lower()}"

    # Add metadata to source table
    df_target = spark.table(target_view)
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=full_source_name,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Get primary columns based on YAML file
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )
    df_target.createOrReplaceTempView(target_view)

    print(f"Final preview for {full_source_name}")
    display(df_target.limit(3))

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, 95, Finished, Available, Finished, False)

Final preview for dbo.Address


SynapseWidget(Synapse.DataFrame, 1fb36769-3853-4618-b350-2cd08d135a34)

Final preview for dbo.Client


SynapseWidget(Synapse.DataFrame, e29d5337-71ad-43e5-b8c8-ff6348714833)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `system_employee_code` cannot be resolved. Did you mean one of the following? [`_lineage_id`, `_source_file`, `_source_system`, `_table`, `_ingest_date`].;
'Project [person_key#40984L, person_number#40985, member_firm_code#40986, person_id#40987, person_status_code#40988, gender#40989, prefix#40990, first_name#40991, first_name_known_as#40992, middle_name#40993, last_name#40994, suffix#40995, initials#40996, photo_path#40997, commencement_date_utc#40998, departure_date_utc#40999, leave_status#41000, team_key#41001, job_title#41002, job_title_pms#41003, practice_group_code#41004, practice_group_key#41005, year_of_call#41006, employee_name_reporting#41007, ... 9 more fields]
+- Project [person_key#40984L, person_number#40985, member_firm_code#40986, person_id#40987, person_status_code#40988, gender#40989, prefix#40990, first_name#40991, first_name_known_as#40992, middle_name#40993, last_name#4

#### Check duplicates

Return duplicates before merge 

In [ ]:
# Check for duplicate hashed keys before merge
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_view = f"tgt_{source_table.lower()}"

    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, -1, Cancelled, , Cancelled, True)

#### Merge to target

Merge to target table based on _hashed_key

In [ ]:
# Merge each final temp view into its target silver table
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    target_view = f"tgt_{source_table.lower()}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Write delta files to path
    df_target = spark.table(target_view)
    load_to_target(df_target, target_table_path, target_load_strategy)

    # Create table based on delta files
    # spark.sql(f"""
    # CREATE TABLE IF NOT EXISTS {full_target_name}
    # USING DELTA
    # LOCATION '{target_table_path}'
    # """)

    # Return operationMetrics > # rows affected
    delta_table = DeltaTable.forPath(spark, target_table_path)
    operation_metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]

    rows_inserted = int(operation_metrics.get("numTargetRowsInserted", operation_metrics.get("numOutputRows", 0)))
    rows_updated = int(operation_metrics.get("numTargetRowsUpdated", 0))
    rows_affected = rows_inserted + rows_updated

    print(full_target_name)
    print(f"rows_affected: {rows_affected}")

StatementMeta(, 26d08c32-f0b8-4e48-9f16-58e97b3dc8b0, -1, Cancelled, , Cancelled, True)